# Rice Leaf Disease Detection — Improved Pipeline

Classes: **Bacterial blight, Brown spot, Leaf smut** ([Kaggle dataset](https://www.kaggle.com/datasets/jay7080dev/rice-plant-diseases-dataset)).

### What changed compared with `01_baseline_cnn_original.ipynb`
| Issue in original | Fix here |
|---|---|
| Train/val/test overlapped: the dataset was reshuffled every epoch, so `take()`/`skip()` gave different images each time and test images leaked into training | Fixed split on disk, made once by `scripts/split_dataset.py` |
| Augmented copies of the same leaf could sit in both train and test | Split script groups near-duplicates (perceptual hashing) and keeps each group in one split |
| `Rescaling(1/300)` | `Rescaling(1/255)` (pixel values are 0–255) |
| `predict()` ignored its `img` argument | Fixed |
| Only accuracy reported | Classification report, per-class F1, confusion matrix |
| Custom CNN only | Added EfficientNetV2-B0 transfer learning + comparison |
| No explainability | Grad-CAM heatmaps |
| No deployment path | Quantized TFLite export for mobile |

**Before running:** create the split once from the project root:
```bash
python scripts/split_dataset.py --src "path/to/rice leaf diseases dataset" --dst data/split
```

## 1. Setup

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
keras.utils.set_random_seed(SEED)

DATA_DIR = "../data/split"          # output of scripts/split_dataset.py
MODELS_DIR = "../models"
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS_BASELINE = 40                 # early stopping usually ends sooner
EPOCHS_HEAD = 10                     # transfer learning, frozen backbone
EPOCHS_FINETUNE = 15                 # transfer learning, top layers unfrozen
PRETRAINED_WEIGHTS = "imagenet"      # set to None to train without downloading weights

os.makedirs(MODELS_DIR, exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Load the fixed splits
Only the training set is shuffled. Val and test stay in a fixed order so the metrics are reproducible.

In [ ]:
def load_split(name, shuffle):
    return keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, name),
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )

train_raw = load_split("train", shuffle=True)
val_raw = load_split("val", shuffle=False)
test_raw = load_split("test", shuffle=False)

class_names = train_raw.class_names
assert class_names == val_raw.class_names == test_raw.class_names, "class folders differ between splits"
NUM_CLASSES = len(class_names)
print(class_names)

In [ ]:
def class_counts(ds):
    labels = np.concatenate([y.numpy() for _, y in ds])
    return np.bincount(labels, minlength=NUM_CLASSES)

counts = {name: class_counts(ds) for name, ds in [("train", train_raw), ("val", val_raw), ("test", test_raw)]}
x = np.arange(NUM_CLASSES)
plt.figure(figsize=(7, 4))
for i, (name, c) in enumerate(counts.items()):
    plt.bar(x + (i - 1) * 0.27, c, width=0.27, label=f"{name} ({c.sum()})")
plt.xticks(x, class_names)
plt.ylabel("images")
plt.title("Images per class and split")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 9))
for images, labels in train_raw.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()

## 3. Input pipeline and augmentation
Augmentation runs only on the training set. Besides flips and rotation it adds zoom, contrast and brightness changes, which mimic phone photos taken under different lighting.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),          # expects 0-255 inputs, applied before rescaling
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (train_raw.cache()
            .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = val_raw.cache().prefetch(AUTOTUNE)
test_ds = test_raw.cache().prefetch(AUTOTUNE)

In [ ]:
# Visual check of the augmentation
plt.figure(figsize=(9, 9))
for images, _ in train_raw.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(augment(images[:1], training=True)[0].numpy().clip(0, 255).astype("uint8"))
        plt.axis("off")
plt.suptitle("Augmented versions of one training image")
plt.show()

## 4. Shared training and evaluation helpers

In [ ]:
def callbacks_for(name):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(os.path.join(MODELS_DIR, f"{name}.keras"), monitor="val_loss", save_best_only=True),
    ]


def plot_history(histories, title):
    acc, val_acc, loss, val_loss = [], [], [], []
    for h in histories:
        acc += h.history["accuracy"]; val_acc += h.history["val_accuracy"]
        loss += h.history["loss"]; val_loss += h.history["val_loss"]
    plt.figure(figsize=(11, 4))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label="train"); plt.plot(val_acc, label="val"); plt.title(f"{title}: accuracy"); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(loss, label="train"); plt.plot(val_loss, label="val"); plt.title(f"{title}: loss"); plt.legend()
    plt.show()


def evaluate(model, ds, title):
    y_true = np.concatenate([y.numpy() for _, y in ds])
    y_prob = model.predict(ds, verbose=0)
    y_pred = y_prob.argmax(axis=1)

    print(f"=== {title} ===")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, cmap="Blues")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xticks(range(NUM_CLASSES), class_names, rotation=30)
    plt.yticks(range(NUM_CLASSES), class_names)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"{title}: confusion matrix")
    plt.colorbar(); plt.tight_layout(); plt.show()

    return {
        "accuracy": float((y_true == y_pred).mean()),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }

## 5. Model A — custom CNN (original architecture, bugs fixed)
Same six conv blocks as before, with three changes: correct `1/255` rescaling, a proper `Input` layer, and dropout to reduce overfitting.

In [ ]:
def build_baseline_cnn():
    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)
    for filters in (32, 32, 32, 64, 64, 64):
        x = layers.Conv2D(filters, 3, activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")


baseline = build_baseline_cnn()
baseline.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
baseline.summary()

In [ ]:
t0 = time.time()
hist_baseline = baseline.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_BASELINE,
                             callbacks=callbacks_for("baseline_cnn"))
baseline_train_time = time.time() - t0
plot_history([hist_baseline], "Custom CNN")

In [ ]:
results = {"Custom CNN": evaluate(baseline, test_ds, "Custom CNN (test set)")}
results["Custom CNN"]["params"] = baseline.count_params()
results["Custom CNN"]["train_min"] = baseline_train_time / 60

## 6. Model B — EfficientNetV2-B0 transfer learning
Two stages:
1. **Head training:** the ImageNet-pretrained backbone is frozen; only the new classifier head learns.
2. **Fine-tuning:** the top of the backbone is unfrozen and trained with a very small learning rate. BatchNorm layers stay frozen so their statistics aren't disturbed.

EfficientNetV2 includes its own preprocessing, so it receives raw 0–255 pixels.

In [ ]:
backbone = keras.applications.EfficientNetV2B0(
    include_top=False,
    weights=PRETRAINED_WEIGHTS,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    include_preprocessing=True,
)
backbone_layers = set(id(l) for l in backbone.layers)

x = layers.GlobalAveragePooling2D(name="gap")(backbone.output)
x = layers.Dropout(0.3, name="head_dropout")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
effnet = keras.Model(backbone.input, outputs, name="efficientnetv2b0_transfer")

# Stage 1: freeze the whole backbone
for layer in backbone.layers:
    layer.trainable = False
effnet.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("trainable params (stage 1):", sum(int(np.prod(w.shape)) for w in effnet.trainable_weights))

In [ ]:
t0 = time.time()
hist_head = effnet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                       callbacks=callbacks_for("efficientnetv2b0"))

In [ ]:
# Stage 2: unfreeze the top ~25% of the backbone, keep BatchNorm frozen
n_unfreeze = len(backbone.layers) // 4
for layer in backbone.layers[-n_unfreeze:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

effnet.compile(optimizer=keras.optimizers.Adam(1e-5),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("trainable params (stage 2):", sum(int(np.prod(w.shape)) for w in effnet.trainable_weights))

hist_ft = effnet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
                     callbacks=callbacks_for("efficientnetv2b0"))
effnet_train_time = time.time() - t0
plot_history([hist_head, hist_ft], "EfficientNetV2-B0")

In [ ]:
results["EfficientNetV2-B0"] = evaluate(effnet, test_ds, "EfficientNetV2-B0 (test set)")
results["EfficientNetV2-B0"]["params"] = effnet.count_params()
results["EfficientNetV2-B0"]["train_min"] = effnet_train_time / 60

## 7. Comparison
Copy this table into the README once you have run the notebook.

In [ ]:
print(f"{'Model':<22}{'Accuracy':>10}{'Macro F1':>10}{'Params':>12}{'Train (min)':>13}")
for name, r in results.items():
    print(f"{name:<22}{r['accuracy']:>10.4f}{r['macro_f1']:>10.4f}{r['params']:>12,}{r['train_min']:>13.1f}")

## 8. Sample predictions (fixed `predict` function)

In [ ]:
def predict(model, img):
    """img: a single HxWx3 image with 0-255 pixel values."""
    img_array = tf.expand_dims(tf.convert_to_tensor(img, dtype=tf.float32), 0)
    probs = model.predict(img_array, verbose=0)[0]
    return class_names[int(np.argmax(probs))], round(100 * float(np.max(probs)), 2)


best_model = effnet if results["EfficientNetV2-B0"]["macro_f1"] >= results["Custom CNN"]["macro_f1"] else baseline
print("Using:", best_model.name)

plt.figure(figsize=(11, 12))
for images, labels in test_raw.take(1):
    for i in range(min(9, len(images))):
        img = images[i].numpy()
        predicted, confidence = predict(best_model, img)
        actual = class_names[labels[i]]
        plt.subplot(3, 3, i + 1)
        plt.imshow(img.astype("uint8"))
        plt.title(f"Actual: {actual}\nPredicted: {predicted}\nConfidence: {confidence}%",
                  color="green" if actual == predicted else "red")
        plt.axis("off")
plt.show()

## 9. Explainability — Grad-CAM
Grad-CAM highlights the image regions that drove the prediction. If the heatmap sits on the lesions, the model has learned the right cues. If it sits on the background or edges, the model is using shortcuts.

In [ ]:
def grad_cam(model, img, last_conv_layer="top_activation"):
    grad_model = keras.Model(model.inputs, [model.get_layer(last_conv_layer).output, model.output])
    x = tf.expand_dims(tf.convert_to_tensor(img, dtype=tf.float32), 0)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(x, training=False)
        class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (img.shape[0], img.shape[1]))[..., 0]
    return cam.numpy(), class_idx


plt.figure(figsize=(12, 8))
for images, labels in test_raw.take(1):
    for i in range(min(6, len(images))):
        img = images[i].numpy()
        cam, idx = grad_cam(effnet, img)
        plt.subplot(2, 3, i + 1)
        plt.imshow(img.astype("uint8"))
        plt.imshow(cam, cmap="jet", alpha=0.4)
        plt.title(f"Actual: {class_names[labels[i]]}\nPred: {class_names[idx]}")
        plt.axis("off")
plt.suptitle("Grad-CAM (EfficientNetV2-B0)")
plt.show()

## 10. Export a quantized TFLite model for mobile

In [ ]:
export_dir = os.path.join(MODELS_DIR, "efficientnetv2b0_savedmodel")
effnet.export(export_dir)

converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]       # dynamic-range INT8 weights
tflite_model = converter.convert()
tflite_path = os.path.join(MODELS_DIR, "efficientnetv2b0_int8.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

keras_size = os.path.getsize(os.path.join(MODELS_DIR, "efficientnetv2b0.keras")) / 1e6
print(f"Keras model:  {keras_size:.1f} MB")
print(f"TFLite INT8:  {len(tflite_model) / 1e6:.1f} MB")

In [ ]:
# Check the TFLite model agrees with the Keras model on the test set
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]

y_true, y_pred = [], []
for images, labels in test_raw:
    for img, label in zip(images.numpy(), labels.numpy()):
        interpreter.set_tensor(inp["index"], img[None].astype(np.float32))
        interpreter.invoke()
        y_pred.append(int(np.argmax(interpreter.get_tensor(out["index"])[0])))
        y_true.append(int(label))
print(f"TFLite test accuracy: {np.mean(np.array(y_true) == np.array(y_pred)):.4f}")

## Next steps
- Stratified 5-fold cross-validation (by duplicate group) to report mean ± std.
- Add a **Healthy** class and confidence-based "unknown" rejection, plus temperature scaling for calibration.
- Cross-dataset test on field images (e.g. Paddy Doctor) to measure real-world generalisation.
- Compare with a ViT/ConvNeXt and a frozen DINOv2 linear probe.
- Lesion segmentation (SAM) to estimate disease severity as % of leaf area.
- Gradio demo on Hugging Face Spaces.